In [1]:
# Load the used classes
from modules_otft._imports import *
from modules_otft._read_data import ReadData
from modules_otft._model import TFTModel
from modules_otft._grafics import  TFTGraphicsPlot
from modules_otft._menu import TFTMenu

# from modules_otft._global_vars import *
from modules_otft._config import *
from modules_otft._utils import *
from modules_otft._display import display_settings

In [2]:
plot = TFTGraphicsPlot()
menu = TFTMenu()
read = ReadData()

In [3]:
# Read and show the path file Json
settings = enter_with_json_file()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


___________________________________________________________________________________ SETTINGS PRESENT IN THE JSON FILE:___________________________________________________________________________________

| path: /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/datas/Org1/tipo p
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| experimental_data_scale_transfer: A
---------------------------------------------------------------------------

In [4]:
ld_voltages = get_load_voltages(settings)

type_curve_plot = get_type_plot(settings)

shift_list = calculate_shift_list(settings)

list_tension_shift = get_shift_list(read, settings)

path_voltages_nominal = read.read_files_experimental(settings['path'])
path_voltages = apply_effective_voltages_to_path(path_voltages_nominal, list_tension_shift)
list_tension = extract_nominal_voltages(path_voltages_nominal)

Vv, Id, input_voltage, n_points, count_transfer, count_output = read.load_data(settings['type_read_data_exp'], path_voltages, settings['current_typic'], settings['experimental_data_scale_transfer'], settings['experimental_data_scale_output'], type_curve_plot)


In [5]:
# Caso o Shift seja aplicado 
path_voltages, new_values_tension, new_list_tension = filter_and_load_files(
    read, settings, path_voltages, list_tension_shift, list_tension_nominal=list_tension)

Vv, Id, input_voltage, n_points, count_transfer, count_output = read.load_data(settings['type_read_data_exp'], path_voltages, settings['current_typic'], settings['experimental_data_scale_transfer'], settings['experimental_data_scale_output'], type_curve_plot)
list_tension_shift = new_values_tension
list_tension = new_list_tension


## **Gerar dados Sintéticos**

In [3]:
from modules_otft._generate_synthetic_data import SyntheticFromSettings

In [4]:
Model = TFTModel(list_tension, n_points)


NameError: name 'list_tension' is not defined

In [5]:
# gen = SyntheticFromSettings(settings, model_cls=TFTModel, out_root="synthetic_from_settings")


In [6]:
# # gerar 8 variações sintéticas por cada arquivo experimental encontrado
# generated_index = gen.generate(n_variations_per_file=2, resample_n_points=10000, variation_mode="voltages", voltage_increment=4.0, voltage_max_variation=20, save_csv=True, save_npz=True)


In [11]:
from modules_otft._inferency import prepare_mlp_features, plot_curve_comparison

## **Treinamento**

In [12]:
from modules_otft._mlp_train import MLPModelTrain 

In [13]:
# Definição dos parâmetros do treino
HIDDEN_LAYERS = (3, 3)
MAX_ITERATIONS = 2000       # Máximo de épocas
TEST_SPLIT = 0.2            # 20% para teste/validação
LEARNING_RATE = 0.0001

BASE_PATH = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/'

INDEX_PATH = BASE_PATH + "synthetic_from_settings/index_generated_mlpv1.json"


In [14]:

# Instanciação do Otimizador/Treinador
mlp_trainer = MLPModelTrain(
    index_path=INDEX_PATH,
    hidden_layers=HIDDEN_LAYERS,
    activation='tanh',
    learning_rate=LEARNING_RATE,
    max_iter=MAX_ITERATIONS,
    test_size=TEST_SPLIT,
    model_path='models/mlp_ids_model.pkl',
    load_model=False 
)

print("Estrutura da MLP definida: Input (5 features) -> 256 -> 128 -> Output (ln|Id|)")

Estrutura da MLP definida: Input (5 features) -> 256 -> 128 -> Output (ln|Id|)


In [15]:
# Inicia o processo de treinamento
trained_model, feature_scaler, target_scaler = mlp_trainer.train_model()
mlp_trainer.save_model('models/mlp_v1.pkl')


Carregando índice de dados em: /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/index_generated_mlpv1.json
Padronizando Features (X) e Target (Y)...
Iniciando treinamento da MLP...
Iteration 1, loss = 1.66613281
Iteration 2, loss = 1.34340333
Iteration 3, loss = 1.07422405
Iteration 4, loss = 0.85692275
Iteration 5, loss = 0.67945107
Iteration 6, loss = 0.53460282
Iteration 7, loss = 0.41866039
Iteration 8, loss = 0.32915518
Iteration 9, loss = 0.26458306
Iteration 10, loss = 0.21962819
Iteration 11, loss = 0.18584321
Iteration 12, loss = 0.15748418
Iteration 13, loss = 0.13267097
Iteration 14, loss = 0.11100617
Iteration 15, loss = 0.09232748
Iteration 16, loss = 0.07635287
Iteration 17, loss = 0.06260978
Iteration 18, loss = 0.05058414
Iteration 19, loss = 0.03973694
Iteration 20, loss = 0.02980131
Iteration 21, loss = 0.02145172
Iteration 22, loss = 0.01579511
Iteration 23, loss = 0.01272120
Iteration 24, loss = 0.01115851
Iteration 25, loss = 0.01026176


True

In [16]:
# -----------------------------------------------------------
TEST_CSV_PATH = 'synthetic_from_settings/Org1/tipo_p/transfer/transfer-neg54V_synth_000.csv'
V_FIXED_TEST = -54.0  # Exemplo: tensão fixa usada na curva de teste
IS_TRANSFER_TEST = True 
# -----------------------------------------------------------


In [17]:
# 1. Carregar dados reais do CSV de teste
df_test = pd.read_csv(TEST_CSV_PATH, header=None)
V_real = df_test.iloc[:, 0].values.astype(float)
I_real = df_test.iloc[:, 1].values.astype(float) # Convertendo para microamperes se necessário


In [18]:
# carregar modelo treinado e scaler_X
PATH_MODEL = BASE_PATH + 'models/Mlp_v1/mlp_v1.pkl'
trained_model, feature_scaler, target_scaler = mlp_trainer.load_model(PATH_MODEL)

Modelo e Scalers carregados de /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/Mlp_v1/mlp_v1*


In [19]:

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real,
    V_fixed=V_FIXED_TEST,
    is_transfer=IS_TRANSFER_TEST,
    
    # Passando os objetos treinados: (Função de preparo, Modelo, Scaler)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log", 
    title=f"Previsão MLP vs. Curva Real ({os.path.basename(TEST_CSV_PATH)})"
)

In [20]:
# 1. Definir os metadados do CSV de teste
TEST_CSV_PATH = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/datas/Org1/tipo p/transfer-50V.csv'
V_FIXED_TEST = -50.0    # Tensão fixa desta curva (e.g., Vgs ou Vds)
IS_TRANSFER_TEST = True # True se for transferência, False se for saída


In [21]:

# 2. Carregar a Curva Real (V e I)
df_test = pd.read_csv(TEST_CSV_PATH, header=None)
V_real = df_test.iloc[:, 0].values.astype(float)
I_real = df_test.iloc[:, 1].values.astype(float)

# 3. Executar o Plot de Comparação

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real*1e6,
    V_fixed=V_FIXED_TEST,
    is_transfer=IS_TRANSFER_TEST,
    
    # O argumento mlp_model é a tupla: (função_prep_X, modelo, scaler_X, scaler_y)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log", # Visualize na escala logarítmica para ver o comportamento de sub-limiar
    title=f"Comparação MLP vs CSV Real ({os.path.basename(TEST_CSV_PATH)})"
)